In [ ]:
from __future__ import division, print_function  
import numpy as np  
import pandas as pd  
import matplotlib.pyplot as plt  
import matplotlib.gridspec as gridspec  
from matplotlib import rcParams  
from matplotlib.ticker import MaxNLocator  
import seaborn as sns  
import os  

# Set up matplotlib parameters to match Cell paper style  
rcParams['font.family'] = 'sans-serif'  
rcParams['font.sans-serif'] = ['Arial']  
rcParams['font.size'] = 10 
rcParams['axes.linewidth'] = 0.8  
rcParams['xtick.major.width'] = 0.8  
rcParams['ytick.major.width'] = 0.8  
rcParams['xtick.major.size'] = 3  
rcParams['ytick.major.size'] = 3  
rcParams['pdf.fonttype'] = 42  # Ensures text is editable in AI  
rcParams['ps.fonttype'] = 42  
rcParams['svg.fonttype'] = 'none'  
rcParams['figure.dpi'] = 300  

# Cell paper color palette  
cell_blue = "#3C5488FF"
cell_orange = "#f9a65a"  
cell_text = "k"  
cell_gray = "#bbb"  



In [ ]:
def calculate_metrics(counts_array, threshold=1):  
    """Calculate multiple statistical metrics for a counts array."""  
    # Handle empty arrays  
    if len(counts_array) == 0 or np.sum(counts_array) == 0:  
        return {  
            'coverage': 0,  
            'gini': 0,  
            'mean_count': 0,  
            'median_count': 0,  
            'zero_count': 100,  
            'total_reads': 0,  
            'ratio_90_10': 0  
        }  
        
    # Calculate coverage  
    coverage = np.sum(counts_array >= threshold) / float(len(counts_array)) * 100  
    
    # Calculate Gini coefficient  
    sorted_array = np.sort(counts_array)  
    cum_array = np.cumsum(sorted_array)  
    n = len(counts_array)  
    index = np.arange(1, n + 1)  
    gini = ((2 * np.sum(index * sorted_array)) / (n * np.sum(counts_array))) - ((n + 1) / n)  
    
    # Calculate 90:10 ratio  
    perc90 = np.percentile(counts_array, 90)  
    perc10 = np.percentile(counts_array, 10)  
    ratio_90_10 = float('inf') if perc10 == 0 else perc90 / perc10  
    
    # Other basic statistics  
    mean_count = np.mean(counts_array)  
    median_count = np.median(counts_array)  
    zero_count = np.sum(counts_array == 0) / float(len(counts_array)) * 100  
    total_reads = np.sum(counts_array)  
    
    return {  
        'coverage': coverage,  
        'gini': gini,  
        'mean_count': mean_count,  
        'median_count': median_count,  
        'zero_count': zero_count,  
        'total_reads': total_reads,  
        'ratio_90_10': ratio_90_10  
    }  

def analyze_screens_from_df(df, count_threshold=1, id_column=None):  
    """Analyze sgRNA count data from a DataFrame where columns are screens."""  
    # Identify the columns containing sgRNA counts  
    count_columns = [col for col in df.columns if col != id_column] if id_column else df.columns  
    
    print("Analyzing {0} screens...".format(len(count_columns)))  
    
    # Results storage  
    results = {  
        'screen': [],  
        'coverage': [],  
        'gini': [],  
        'mean_count': [],  
        'median_count': [],  
        'zero_count': [],  
        'total_reads': [],  
        'ratio_90_10': []  
    }  
    
    # Analyze each screen  
    for column in count_columns:  
        counts = df[column].values  
        # Skip empty columns  
        if np.sum(counts) == 0:  
            print("Skipping empty column: {0}".format(column))  
            continue  
            
        # Calculate metrics  
        metrics = calculate_metrics(counts, threshold=count_threshold)  
        
        # Store results  
        results['screen'].append(column)  
        for key in metrics:  
            results[key].append(metrics[key])  
    
    # Convert to DataFrame  
    results_df = pd.DataFrame(results)  
    
    return results_df  

def plot_correlation_heatmap(counts_df, id_column=None, output_dir='.'):  
    """Plot correlation heatmap between screens."""  
    # Identify count columns  
    count_columns = [col for col in counts_df.columns if col != id_column] if id_column else counts_df.columns  
    
    # Calculate correlation matrix  
    corr_matrix = counts_df[count_columns].corr()  
    # Create mask for the upper triangle  
    mask = np.zeros_like(corr_matrix, dtype=bool)  
    mask[np.triu_indices_from(mask, k=1)] = True  # k=1 to exclude diagonal  
    
    # Create figure  
    plt.figure(figsize=(8, 7))  
    
    # Plot heatmap with red colormap and masked upper triangle  
    sns.heatmap(  
        corr_matrix,   
        cmap='YlOrRd',  # Red colormap  
        square=True,  
        linewidths=0.5,  
        cbar_kws={"shrink": 0.75},  
        vmin = 0,  
        mask=mask  # Apply mask to show only lower triangle  
    )  

    # Customize plot  
    plt.title('Pearson Correlation Between Screens', fontsize=10, color=cell_text)  
    plt.tight_layout()  
    
    # Save figure  
    corr_matrix.to_csv(os.path.join(output_dir, 'E_correlation.csv'))
    plt.savefig(os.path.join(output_dir, 'E_correlation_heatmap.pdf'), dpi=300, bbox_inches='tight')  
    plt.savefig(os.path.join(output_dir, 'E_correlation_heatmap.png'), dpi=300, bbox_inches='tight')  
    plt.close()  
    
    # Create pairwise scatter plots for up to 6 screens  
    if len(count_columns) <= 6:  
        # Set up scatterplot grid  
        g = sns.pairplot(  
            counts_df[count_columns],  
            diag_kind='kde',  
            plot_kws={'alpha': 0.5, 'edgecolor': 'none', 's': 3, 'c': 'k'},  
            corner=True  # Only show lower triangle  
        )  
        g.fig.suptitle('Pairwise Screen Comparisons', y=1.02, fontsize=10, color=cell_text)  
        g.fig.tight_layout()  
        
        # Save figure  
        g.savefig(os.path.join(output_dir, 'F_pairwise_scatterplots.pdf'), dpi=300, bbox_inches='tight')  
        g.savefig(os.path.join(output_dir, 'F_pairwise_scatterplots.png'), dpi=300, bbox_inches='tight')  
        plt.close()  
    
    # Create log-log scatterplot matrix for more detailed view  
    if len(count_columns) <= 8:  
        fig, axes = plt.subplots(  
            len(count_columns), len(count_columns),   
            figsize=(10, 10),   
            sharex='col',   
            sharey='row'  
        )  
        
        # Add minimum value to avoid log(0)  
        min_nonzero = np.min(counts_df[count_columns].values[counts_df[count_columns].values > 0])  
        log_df = counts_df[count_columns].copy() + min_nonzero/10  
        
        # Plot each comparison  
        for i, screen1 in enumerate(count_columns):  
            for j, screen2 in enumerate(count_columns):  
                ax = axes[i, j]  
                
                if i == j:  # Diagonal: show histogram  
                    ax.hist(np.log10(log_df[screen1]), bins=30, alpha=0.7, color=cell_blue)  
                    if i == 0:  
                        ax.set_title(screen1, fontsize=7, color=cell_text)  
                    if j == 0:  
                        ax.set_ylabel(screen1, fontsize=7, color=cell_text)  
                    
                else:  # Off-diagonal: show scatterplot  
                    ax.scatter(  
                        np.log10(log_df[screen2]),   
                        np.log10(log_df[screen1]),  
                        s=2,   
                        alpha=0.5,   
                        color=cell_orange  
                    )  
                    
                    # Add correlation coefficient  
                    corr = np.corrcoef(log_df[screen2], log_df[screen1])[0,1]  
                    ax.text(  
                        0.05, 0.95,   
                        "r={:.2f}".format(corr),  
                        transform=ax.transAxes,  
                        va='top',  
                        fontsize=5,  
                        color=cell_text  
                    )  
                    
                    if i == len(count_columns)-1:  
                        ax.set_xlabel(screen2, fontsize=7, color=cell_text)  
                    if j == 0:  
                        ax.set_ylabel(screen1, fontsize=7, color=cell_text)  
                
                # Remove top and right spines  
                ax.spines['top'].set_visible(False)  
                ax.spines['right'].set_visible(False)  
        
        plt.suptitle('Log-Log Scatterplot Matrix', fontsize=10, color=cell_text)  
        plt.tight_layout(rect=[0, 0, 1, 0.97])  
        
        plt.savefig(os.path.join(output_dir, 'G_loglog_scatterplot_matrix.pdf'), dpi=300, bbox_inches='tight')  
        plt.savefig(os.path.join(output_dir, 'G_loglog_scatterplot_matrix.png'), dpi=300, bbox_inches='tight')  
        plt.close()  
        
    return  

def plot_metrics(results_df, counts_df=None, id_column=None, output_dir='.'):  
    """Create Gini-style bar plots for metrics, saving each panel separately."""  
    if not os.path.exists(output_dir):  
        os.makedirs(output_dir)  

    def gini_style_barplot(metric, title, ylabel, ylim=None, value_format="{0:.2f}"):  
        fig, ax = plt.subplots(figsize=(3, 3))  
        screen_order = results_df['screen'].tolist()  
        bar_colors = [cell_orange] + [cell_blue] * (len(results_df) - 1)  

        bars = ax.bar(  
            range(len(results_df)),  
            results_df[metric],  
            color=bar_colors,  
            edgecolor='k',  
            linewidth=0.5,  
            alpha=1,  
            width=0.8  
        )  

        # Add values on top of bars  
        for i, bar in enumerate(bars):  
            ax.text(  
                bar.get_x() + bar.get_width()/2,  
                bar.get_height() + (ylim[1] - ylim[0]) * 0.01 if ylim else bar.get_height() + 0.01,  
                value_format.format(results_df[metric].iloc[i]),  
                ha='center', fontsize=8, color=cell_text  
            )  

        ax.set_xlabel('Screen', fontsize=9, color=cell_text)  
        ax.set_ylabel(ylabel, fontsize=9, color=cell_text)  
        ax.set_title(title, fontsize=10, color=cell_text)  
        if ylim:  
            ax.set_ylim(ylim)  
        ax.set_xticks(range(len(results_df)))  
        ax.set_xticklabels(screen_order, fontsize=8, rotation=45, ha='right')  
        ax.grid(True, axis='y', linestyle='--', alpha=0.5, color=cell_gray)  
        ax.set_axisbelow(True)  
        ax.spines['top'].set_visible(True)  
        ax.spines['right'].set_visible(True)  
        ax.spines['left'].set_color(cell_text)  
        ax.spines['bottom'].set_color(cell_text)  
        plt.tight_layout()  
        for spine_name in ['left','bottom','top','right']:  
            ax.spines[spine_name].set_color(cell_text)  
            ax.spines[spine_name].set_visible(True)  
            ax.spines[spine_name].set_linewidth(0.7)  # Make spines thinner here 
        return fig, ax  

    # Panel C: Gini-style Coverage Barplot  
    fig, ax = gini_style_barplot(  
        metric='coverage',  
        title='sgRNA Coverage by Screen',  
        ylabel='Coverage (%)',  
        ylim=(0, 120),  
        value_format="{0:.1f}"  
    )  
    plt.savefig(os.path.join(output_dir, 'A_coverage_barplot.pdf'), dpi=300, bbox_inches='tight')  
    plt.savefig(os.path.join(output_dir, 'A_coverage_barplot.png'), dpi=300, bbox_inches='tight')  
    plt.close()  

    # Panel D: Gini Barplot (already in this style)  
    fig, ax = gini_style_barplot(  
        metric='gini',  
        title='sgRNA Distribution Inequality',  
        ylabel='Gini Coefficient',  
        ylim=(0, 1),  
        value_format="{0:.2f}"  
    )  
    plt.savefig(os.path.join(output_dir, 'B_gini_barplot.pdf'), dpi=300, bbox_inches='tight')  
    plt.savefig(os.path.join(output_dir, 'B_gini_barplot.png'), dpi=300, bbox_inches='tight')  
    plt.close()  

    # Panel: Gini-style 90:10 Ratio Barplot  
    fig, ax = gini_style_barplot(  
        metric='ratio_90_10',  
        title='sgRNA Distribution Inequality (90:10 Ratio)',  
        ylabel='90:10 Ratio',  
        ylim=(0, 10),  
        value_format="{0:.2f}"  
    )  
    plt.savefig(os.path.join(output_dir, 'C_ratio_90_10_barplot.pdf'), dpi=300, bbox_inches='tight')  
    plt.savefig(os.path.join(output_dir, 'C_ratio_90_10_barplot.png'), dpi=300, bbox_inches='tight')  
    plt.close()  

    # Panel E: Lorenz Curves stays unchanged  
    if counts_df is not None:  
        fig, ax = plt.subplots(figsize=(3, 3))  
        count_columns = [col for col in counts_df.columns if col != id_column] if id_column else counts_df.columns  
        colors = sns.color_palette("husl", len(count_columns))  
        ax.plot([0, 1], [0, 1], '--', color='k', label='Line of Equality', linewidth=1)  
        for i, screen in enumerate(count_columns):  
            counts = counts_df[screen].values  
            counts = counts[counts >= 0]  # Remove negative values if any  
            sorted_counts = np.sort(counts)  
            cum_counts = np.cumsum(sorted_counts)  
            lorenz_curve = cum_counts / cum_counts[-1] if cum_counts[-1] > 0 else np.zeros_like(cum_counts)  
            lorenz_x = np.arange(1, len(counts) + 1) / len(counts)  
            ax.plot(  
                lorenz_x,  
                lorenz_curve,  
                label="{0} (Gini={1:.3f})".format(  
                    screen, results_df[results_df['screen'] == screen]['gini'].values[0]  
                ),  
                color=colors[i],  
                linewidth=1,  
                alpha=0.9  
            )  
        ax.set_xlim(0, 1)  
        ax.set_ylim(0, 1)  
        ax.set_xlabel('Cumulative Fraction of sgRNAs', fontsize=10, color=cell_text)  
        ax.set_ylabel('Cumulative Fraction of Counts', fontsize=10, color=cell_text)  
        ax.set_title('Lorenz Curves of sgRNA Distributions', fontsize=10, color=cell_text)  
        ax.grid(True, linestyle='--', alpha=0.5, color=cell_gray)  
        ax.set_axisbelow(True)  
        ax.spines['top'].set_visible(True)  
        ax.spines['right'].set_visible(True)  
        ax.spines['top'].set_position(('data', 1))  
        ax.spines['right'].set_position(('data', 1))  
        ax.spines['top'].set_color(cell_text)  
        ax.spines['right'].set_color(cell_text)  
        ax.spines['left'].set_color(cell_text)  
        ax.spines['bottom'].set_color(cell_text)  
        ax.legend(  
            loc='upper right', bbox_to_anchor=(2, 1),  
            fontsize=5,  
            frameon=True,  
            framealpha=0.9,  
            edgecolor=cell_gray  
        )  
        plt.tight_layout()  
        plt.savefig(os.path.join(output_dir, 'D_lorenz_curves.pdf'), dpi=300, bbox_inches='tight')  
        plt.savefig(os.path.join(output_dir, 'D_lorenz_curves.png'), dpi=300, bbox_inches='tight')  
        plt.close()  

    print(f"Individual panels saved to {output_dir}")  
    return 

def analyze_sgrna_counts_enhanced(counts_df, id_column=None, count_threshold=1, output_dir='sgRNA_analysis'):  
    """Complete analysis with individual panels and correlation metrics."""  
    if not os.path.exists(output_dir):  
        os.makedirs(output_dir)  
        
    # Analyze the screens  
    results = analyze_screens_from_df(counts_df, count_threshold=count_threshold, id_column=id_column)  
    
    # Print summary  
    print("\nSummary Statistics:")  
    print(results)  
    
    # Create plots as individual panels  
    plot_metrics(results, counts_df=counts_df, id_column=id_column, output_dir=output_dir)  
    
    # Add correlation plots  
    plot_correlation_heatmap(counts_df, id_column=id_column, output_dir=output_dir)  
    
    print("\nAnalysis complete!")  
    return results  

In [ ]:
counts_path = '../counts/'
QC_path = '../QC/'

for counts_file in os.listdir(counts_path):
    if counts_file.endswith('.txt'):
        out = QC_path + counts_file.split('.')[0]
        if counts_file.split('.')[0] not in os.listdir(QC_path):
            os.makedirs(out)
        df=pd.read_table(counts_path+counts_file)
        if 'GW' in counts_file:
            df_ori = pd.read_csv('../counts/GW_plasmid_library.csv')
        elif 'ND' in counts_file:
            df_ori = pd.read_csv('../counts/ND_plasmid_library.csv')
        df = df.merge(df_ori, left_on = 'Guide', right_on = 'sgRNA', how = 'outer').drop('Guide', axis=1)

        df = df.fillna(0)

        df = df.drop('Gene', axis=1)
        df = df.loc[:,[df.columns[-1]]+df.columns[:-1].tolist()]
        df = df.rename({'counts':'library'},axis=1)
        df = df.set_index('sgRNA')
        analyze_sgrna_counts_enhanced(df, output_dir=out)  
